In [1]:
import numpy as np 
import pandas as pd 
import h5py
import re
import uproot
import glob 
import copy, sys, os
import matplotlib.pyplot as plt
from tqdm import tqdm 
# adding path to folder
import sys
sys.path.append('/global/homes/g/gkufatty/projects/neutron-multiplicity/analysis')
from utils_analysis import ParticleCode, load_dataset

# Load your custom style 
plt.style.use('/global/homes/g/gkufatty/projects/neutron-multiplicity/analysis/cfg/my_custom_plot.mplstyle')

# use these lines on top of your matplotlib script
import matplotlib.ticker
class MyLocator(matplotlib.ticker.AutoMinorLocator):
    def __init__(self, n=4):
        super().__init__(n=n)
matplotlib.ticker.AutoMinorLocator = MyLocator        
 
# Now use matplotlib as usual.       
import matplotlib.pyplot as plt
plt.rcParams["xtick.minor.visible"] =  True
plt.rcParams["ytick.minor.visible"] =  True

sys.path.insert(0, '/global/homes/g/gkufatty/projects/neutron-multiplicity/tools/')
from caf_readers import CafReader
from mx2_matching import Mx2_DS_Match


In [2]:
# Load datasets 
n_files = 100
location = "nersc"
type="mr6"
df = load_dataset(n_files,location, type)
# Create a caf reader
caf_reader = CafReader(df)
pdg_tab = ParticleCode()

Reading  100  files gk
Location selected nersc
Openning MiniRun 6 CAFs


  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [02:42<00:00,  1.62s/it]


In [3]:
# Load datasets 
n_files = 200
location = "nersc"
type="mr6"
dff = load_dataset(n_files,location, type)
# Create a caf reader
# caf_reader = CafReader(df)
# pdg_tab = ParticleCode()

Reading  200  files gk
Location selected nersc
Openning MiniRun 6 CAFs


100%|██████████| 200/200 [04:59<00:00,  1.50s/it]


In [4]:
#dimensions of the detector
anode_xs = np.array([-63.931, -3.069, 3.069, 63.931])
anode_ys = np.array([-19.8543, 103.8543]) - 42  # Subtract 42 from all y coordinates
anode_zs = np.array([-64.3163, -2.6837, 2.6837, 64.3163])
#Fiducial Volume
tpc_dist = 5
xbound = 63.931
ybound = 62.076
zbound = 64.3163

def track_selection_signal(ev_data,ev):

    minerva_data = caf_reader.get_minerva_data(df.iloc[ev])

    x_start_track  = ev_data['rec.nd.lar.dlp.tracks.start.x']
    y_start_track = ev_data['rec.nd.lar.dlp.tracks.start.y']
    z_start_track = ev_data['rec.nd.lar.dlp.tracks.start.z']

    x_end_track = ev_data['rec.nd.lar.dlp.tracks.end.x']
    y_end_track = ev_data['rec.nd.lar.dlp.tracks.end.y']
    z_end_track = ev_data['rec.nd.lar.dlp.tracks.end.z']

    x_coords_vtx = ev_data['rec.common.ixn.dlp.vtx.x']
    y_coords_vtx = ev_data['rec.common.ixn.dlp.vtx.y']
    z_coords_vtx = ev_data['rec.common.ixn.dlp.vtx.z']

    # Stack the coordinates into a single 2D array
    position_vtx = np.vstack((x_coords_vtx, y_coords_vtx, z_coords_vtx)).T
    start_track = np.vstack((x_start_track, y_start_track, z_start_track)).T

    track_on_vtx = np.zeros_like(ev_data['rec.common.ixn.dlp.vtx.x'], dtype=bool)
    track_out = np.zeros_like(ev_data['rec.common.ixn.dlp.vtx.x'], dtype=bool)
    track_Mx2 = np.zeros_like(ev_data['rec.common.ixn.dlp.vtx.x'], dtype=bool)

    for vtx in range(len(ev_data['rec.common.ixn.dlp.vtx.x'])):
        n_reco_tracks = ev_data['rec.nd.lar.dlp.tracks..length'][vtx]
        n_pre = np.sum(ev_data['rec.nd.lar.dlp.tracks..length'][:vtx]) if vtx > 0 else 0
        
        vertex_position = position_vtx[vtx]
        # Scan over particles associated with this interaction
        found_match = False
        
        for ip in range(n_pre, n_pre + n_reco_tracks):
            if np.array_equal(vertex_position, start_track[ip]):
                found_match = True
                #if the track is found, then check if the track also goes out the boundary
                if (z_end_track[ip] >= zbound):
                    track_out[vtx] = True 
                    muon_track = [[x_start_track[ip], y_start_track[ip],z_start_track[ip]],[x_end_track[ip],y_end_track[ip],z_end_track[ip]]]
                    bm, overlap, dx, dy, exit_minerva = Mx2_DS_Match(muon_track,minerva_data)
                    if (exit_minerva == True) and (bm is not None):
                        track_Mx2[vtx] == True

        track_on_vtx[vtx] = found_match
    return track_on_vtx,  track_out, track_Mx2

In [5]:
ev_data=df.iloc[20]
n_ixn= len(ev_data['rec.common.ixn.dlp.vtx.z'])

print(f'This event has {n_ixn} reco vertices')
for ixn_index in range(n_ixn):
    print(f'Reco vertex {ixn_index}')
    n_int = ev_data['rec.common.ixn.dlp.truth..length'][ixn_index]
    if(ixn_index==0):
        n_pre = 0
    else: 
        n_pre = np.sum(ev_data['rec.common.ixn.dlp.truth..length'][:ixn_index]) 
    #print(n_pre)
    #print(n_pre,n_pre + n_int)
    max_overlap=0
    for ip in range(n_pre,n_pre + n_int):
        #print(ip)
        print(n_pre,n_pre + n_int  )
        temp_overlap = ev_data['rec.common.ixn.dlp.truthOverlap'][ip]
        temp_tp_ixn = ev_data['rec.common.ixn.dlp.truth'][ip]
        #print(f'this vtx has tem overlap {temp_overlap} in index {temp_tp_ixn}')
        if (temp_overlap> max_overlap):
            max_overlap = temp_overlap
            best_match_idx = temp_tp_ixn
        print(f'best overlap is at index {best_match_idx} with {max_overlap}')
# ev_data['rec.common.ixn.dlp.vtx.z']

This event has 3 reco vertices
Reco vertex 0
0 1
best overlap is at index 0 with 0.9795501232147217
Reco vertex 1
1 2
best overlap is at index 1 with 1.0
Reco vertex 2
2 4
best overlap is at index 2 with 0.9750000238418579
2 4
best overlap is at index 2 with 0.9750000238418579


In [49]:
def calculate_cosL(ev_data, ip):
    dz = ev_data['rec.mc.nu.prim.start_pos.z'][ip] - ev_data['rec.mc.nu.prim.end_pos.z'][ip]
    start_pos=(
        ev_data['rec.mc.nu.prim.start_pos.x'][ip], 
        ev_data['rec.mc.nu.prim.start_pos.y'][ip],
        ev_data['rec.mc.nu.prim.start_pos.z'][ip]
        ),
    end_pos=(
        ev_data['rec.mc.nu.prim.end_pos.x'][ip], 
        ev_data['rec.mc.nu.prim.end_pos.y'][ip],
        ev_data['rec.mc.nu.prim.end_pos.z'][ip]
        )

    track_length = np.linalg.norm(np.array(start_pos) - np.array(end_pos))
    cosL=  abs(dz/track_length)
    return cosL

In [1]:
# count=0
# acount=0
# m=0
# cc=0


# for ev in range(len(df)):
#     ev_data=df.iloc[ev]
#     signal =  np.zeros_like(ev_data['rec.mc.nu.vtx.x'], dtype=bool)

#     mask_fv_t = (
#     (abs(ev_data['rec.mc.nu.vtx.x'])<xbound-tpc_dist) &
#     (abs(ev_data['rec.mc.nu.vtx.x'])>tpc_dist) &
#     (abs(ev_data['rec.mc.nu.vtx.y'])<ybound-tpc_dist) &
#     (abs(ev_data['rec.mc.nu.vtx.z'])>tpc_dist) &
#     (abs(ev_data['rec.mc.nu.vtx.z'])<zbound-tpc_dist) &
#     (ev_data['rec.mc.nu.targetPDG']== pdg_tab.argon) &
#     (ev_data['rec.mc.nu.iscc']==1) &
#     (np.abs(ev_data['rec.mc.nu.pdg']) == pdg_tab.numu)  
#     )

#     signal[mask_fv_t]=True

#     for ix in range(ev_data['rec.mc.nu..length']):
#          if (mask_fv_t[ix]==True):
#             n_particles = ev_data['rec.mc.nu.prim..length'][ix]
#             if(ix==0):
#                 n_pre = 0
#             else: 
#                 n_pre = np.sum(ev_data['rec.mc.nu.prim..length'][:ix]) 
#             for ip in range(n_pre,n_pre + n_particles):
#                 pdg= ev_data['rec.mc.nu.prim.pdg'][ip]
                
#                 if pdg == pdg_tab.muon: 
#                     m+=1
#                     Elep= ev_data['rec.mc.nu.prim.p.E'][ip]
#                     cosL=calculate_cosL(ev_data,ip)
#                     if (cosL<0.9 and Elep<1):
#                         signal[ix] = False


In [86]:

num_neutrinos_fv_r=0 # in fv
num_neutrinos_fv_t=0 # in fv
num_neutrinos_signal_t=0
#num_neutrinos_fv_tt =0

#for tracks
num_track_vtx_out_fv=0 #produced track start on vtx and exit 2x2 downstream
num_track_Mx2_fv = 0 #num of tracks that crossed minerva 

count_reco=0
count_match_FV=0

filtered_data = []


for ev in range(len(df)):
    ev_data=df.iloc[ev]
    signal =  np.zeros_like(ev_data['rec.mc.nu.vtx.x'], dtype=bool)

    mask_fv_t = (
    (abs(ev_data['rec.mc.nu.vtx.x'])<xbound-tpc_dist) &
    (abs(ev_data['rec.mc.nu.vtx.x'])>tpc_dist) &
    (abs(ev_data['rec.mc.nu.vtx.y'])<ybound-tpc_dist) &
    (abs(ev_data['rec.mc.nu.vtx.z'])>tpc_dist) &
    (abs(ev_data['rec.mc.nu.vtx.z'])<zbound-tpc_dist) &
    (ev_data['rec.mc.nu.targetPDG']== pdg_tab.argon) &
    (ev_data['rec.mc.nu.iscc']==1) &
    (np.abs(ev_data['rec.mc.nu.pdg']) == pdg_tab.numu)  
    )

    signal[mask_fv_t]=True

    for ix in range(ev_data['rec.mc.nu..length']):
         if (mask_fv_t[ix]==True):
            n_particles = ev_data['rec.mc.nu.prim..length'][ix]
            if(ix==0):
                n_pre = 0
            else: 
                n_pre = np.sum(ev_data['rec.mc.nu.prim..length'][:ix]) 
            for ip in range(n_pre,n_pre + n_particles):
                pdg= ev_data['rec.mc.nu.prim.pdg'][ip]
                
                if pdg == pdg_tab.muon: 
                    m+=1
                    Elep= ev_data['rec.mc.nu.prim.p.E'][ip]
                    cosL=calculate_cosL(ev_data,ip)
                    if (cosL<0.9 and Elep<1):
                        signal[ix] = False

    mask_fv_r = (
        (abs(ev_data['rec.common.ixn.dlp.vtx.x'])<xbound-tpc_dist) &
        (abs(ev_data['rec.common.ixn.dlp.vtx.x'])>tpc_dist) &
        (abs(ev_data['rec.common.ixn.dlp.vtx.y'])<ybound-tpc_dist) &
        (abs(ev_data['rec.common.ixn.dlp.vtx.z'])>tpc_dist) &
        (abs(ev_data['rec.common.ixn.dlp.vtx.z'])<zbound-tpc_dist)
    )
    
    num_neutrinos_fv_r += np.sum(mask_fv_r)
    num_neutrinos_fv_t += np.sum(mask_fv_t)
    num_neutrinos_signal_t+= np.sum(signal)
    
    for ixn_index, value in enumerate(mask_fv_r):
        if value:  
            count_reco +=1 #count num of vertices in fv
            n_int = ev_data['rec.common.ixn.dlp.truth..length'][ixn_index]
            if(ixn_index==0):
                n_pre = 0
            else: 
                n_pre = np.sum(ev_data['rec.common.ixn.dlp.truth..length'][:ixn_index]) 
            max_overlap=0
            best_match_idx=None
            for ip in range(n_pre,n_pre + n_int):
                temp_overlap = ev_data['rec.common.ixn.dlp.truthOverlap'][ip]
                temp_tp_ixn = ev_data['rec.common.ixn.dlp.truth'][ip]
                #print(f'this vtx has tem overlap {temp_overlap} in index {temp_tp_ixn}')
                if (temp_overlap> max_overlap):
                    max_overlap = temp_overlap
                    best_match_idx = temp_tp_ixn
            
            if best_match_idx!=None:
                delta_x = np.abs(ev_data['rec.mc.nu.vtx.x'][best_match_idx] - ev_data['rec.common.ixn.dlp.vtx.x'][ixn_index])
                delta_y = np.abs(ev_data['rec.mc.nu.vtx.y'][best_match_idx] - ev_data['rec.common.ixn.dlp.vtx.y'][ixn_index])
                delta_z = np.abs(ev_data['rec.mc.nu.vtx.z'][best_match_idx] - ev_data['rec.common.ixn.dlp.vtx.z'][ixn_index])
                if ((signal[best_match_idx]==True) and 
                    delta_x < 5 and
                    delta_y < 5 and
                    delta_z < 5):
                        count_match_FV+=1
                        filtered_data.append(ev_data)
                    #print('contained')

        else: continue

                #print(f'best overlap is at index {best_match_idx} with {max_overlap}')
        

In [87]:
print(f'there are {count_reco} reco vertices in FV, and {count_match_FV} vertices with true interaction match and {num_neutrinos_signal_t} true interactions in FV')

there are 7008 reco vertices in FV, and 1164 vertices with true interaction match and 1239 true interactions in FV


In [89]:
cou nt_match_FV/count_reco,  count_match_FV/num_neutrinos_signal_t

(0.1660958904109589, 0.9394673123486683)